# Experimento 03 — Selección de modelos robusta al shift temporal

Este cuaderno repite el pipeline enriquecido del experimento 02 (baseline + distancia geográfica) pero cambia el criterio con el que elegimos campeón: en lugar de la media del score entre los tres pliegues de la validación cruzada temporal usamos el *peor* pliegue. La intuición, prestada de la literatura sobre CV para series temporales (Bergmeir & Benítez, 2012) y del aviso de Varma & Simon (2006) sobre tomar la media como garantía de generalización, es que un modelo cuyo peor pliegue es aceptable debería generalizar mejor cuando la distribución se mueve en el tiempo.

La configuración, los datos y los modelos son los mismos del experimento 02. Lo único nuevo es `selection_criterion='worst'` en `ExperimentConfig`. Los resultados se guardan en `outputs/experimentos/03_seleccion_robusta/`.

## 1. Entorno y datos

Resolvemos la raíz del repo (local o Colab) y aseguramos que los CSV de Olist están en `data/raw/`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('experimentos', 'notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Raíz del repositorio:', root)

Raíz del repositorio: /home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile

raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))

CSV disponibles en data/raw: 9


## 2. Configuración y dataset enriquecido

Misma construcción que el experimento 02 —dataset a nivel de pedido, enriquecido con distancia cliente↔vendedor—, pero fijamos `selection_criterion='worst'`. Las demás constantes (semilla, partición, bootstrap) no cambian.

In [4]:
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.geography import attach_order_geography
from src.experiments import run_task

In [5]:
MAX_ROWS = 15_000  # Use None para la corrida completa.
config = ExperimentConfig(
    max_rows=MAX_ROWS,
    cv_splits=3,
    bootstrap_iterations=500,
    selection_criterion='worst',
)
config

ExperimentConfig(random_state=42, max_rows=15000, cv_splits=3, bootstrap_iterations=500, min_frequency=20, low_cardinality_max=30, high_cardinality_min=100, selection_criterion='worst')

In [6]:
tables = load_olist_tables(raw)
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
coverage = orders['mean_customer_seller_distance_km'].notna().mean()
print(f'Pedidos enriquecidos: {len(orders):,}, cobertura geográfica: {coverage:.2%}')

Pedidos enriquecidos: 99,441, cobertura geográfica: 98.73%


## 3. Regresión — días hasta la entrega

El ranking se ordena ahora por `cv_worst` (el peor pliegue), no por `validation_MAE`. Reportamos las dos columnas de CV (media y peor) para dejar ver cómo se mueve el reparto, no solo quién gana.

In [7]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
X_reg.shape

(96476, 25)

In [8]:
reg = run_task('regression', X_reg, y_reg, dates_reg, config)

/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [9]:
reg['ranking'][['model', 'cv_mean', 'cv_worst', 'validation_MAE', 'test_MAE', 'test_ci95_low', 'test_ci95_high']]

,model,cv_mean,cv_worst,validation_MAE,test_MAE,test_ci95_low,test_ci95_high
0,LinearSVR (SVM),-5.589561,-6.123649,5.748942,4.611700,4.467956,4.744079
1,KNN (no parametrico),-5.961890,-6.622168,6.114772,5.176658,5.017315,5.332214
2,Ridge (parametrico),-5.924355,-6.699971,6.699971,6.345582,6.183139,6.496012
3,Random Forest (ensamble),-6.153236,-6.845352,6.845352,4.855147,4.702110,5.020480
4,Baseline trivial (mediana),-6.915378,-7.876737,6.917828,5.043977,4.905983,5.182219
5,MLP (red neuronal),-7.992064,-12.354568,6.622786,5.543068,5.365551,5.697731


LinearSVR vuelve a ganar. La lectura útil no es quién gana, es cómo se abre el espacio entre `cv_mean` y `cv_worst`: MLP tiene un peor pliegue 55 % por debajo de su media (−12.35 vs −7.99). El criterio robusto convierte esa inestabilidad entre pliegues en un castigo explícito que el criterio por media escondía.

## 4. Clasificación — probabilidad de entrega tardía

El objetivo real del experimento. En los experimentos 01 y 02 el campeón por media de CV (Random Forest) resultó peor que el baseline trivial sobre test, mientras que logística —descartada por la media— llega a 0.73 ROC-AUC en test. La hipótesis: si el criterio castiga la inestabilidad entre pliegues, Logística podría subir en el ranking.

In [10]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
X_cls.shape

(96476, 25)

In [11]:
cls = run_task('classification', X_cls, y_cls, dates_cls, config)

In [12]:
cls['ranking'][['model', 'cv_mean', 'cv_worst', 'validation_ROC_AUC', 'test_ROC_AUC', 'test_ci95_low', 'test_ci95_high']]

,model,cv_mean,cv_worst,validation_ROC_AUC,test_ROC_AUC,test_ci95_low,test_ci95_high
0,Random Forest (ensamble),0.631539,0.563248,0.714865,0.511994,0.470780,0.551532
1,Logistica (parametrico),0.599240,0.557342,0.629721,0.734426,0.703674,0.764691
2,LinearSVC (SVM),0.589980,0.544152,0.622194,0.729154,0.696278,0.759988
3,KNN (no parametrico),0.580595,0.537144,0.648879,0.514497,0.473990,0.553937
4,MLP (red neuronal),0.550509,0.510350,0.628286,0.388919,0.342291,0.435873
5,Baseline trivial (clase mayoritaria),0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


**La hipótesis falla.** Random Forest gana tanto por media (0.632) como por peor pliegue (0.563). Logística queda segunda en los dos criterios (0.599 / 0.557). La diferencia en el peor pliegue es apenas 0.006 ROC-AUC. Random Forest es uniformemente mejor dentro de la ventana de desarrollo.

Y sin embargo, en test, Random Forest cae a 0.512 mientras Logística llega a 0.734. Es la observación clave del experimento: **el shift que importa está entre desarrollo y prueba, no entre pliegues de la CV**. Un criterio robusto sobre los pliegues no lo puede corregir porque ese shift no se manifiesta ahí.

## 5. Comparación con los experimentos anteriores

Reunimos los tres `summary.json` para dejar en una tabla la secuencia completa.

In [13]:
import json
import pandas as pd

def read_summary(name):
    return json.loads((root / f'outputs/experimentos/{name}/summary.json').read_text())

exp01 = read_summary('01_baseline_pre_compra')
exp02 = read_summary('02_distancia_geografica')

comparison = pd.DataFrame(
    [
        {
            'experimento': '01 baseline',
            'regresión MAE': exp01['regression']['test_primary'],
            'clasificación ROC-AUC': exp01['classification']['test_primary'],
        },
        {
            'experimento': '02 + distancia',
            'regresión MAE': exp02['regression']['test_primary'],
            'clasificación ROC-AUC': exp02['classification']['test_primary'],
        },
        {
            'experimento': '03 criterio robusto',
            'regresión MAE': float(reg['ranking'].iloc[0]['test_MAE']),
            'clasificación ROC-AUC': float(cls['ranking'].iloc[0]['test_ROC_AUC']),
        },
    ]
)
comparison

,experimento,regresión MAE,clasificación ROC-AUC
0,01 baseline,4.693817,0.486571
1,02 + distancia,4.611700,0.511994
2,03 criterio robusto,4.611700,0.511994


## 6. Persistencia de resultados

Se guardan las mismas cuatro tablas por tarea y el `summary.json` que incluye el criterio empleado.

In [14]:
output = root / 'outputs' / 'experimentos' / '03_seleccion_robusta'
output.mkdir(parents=True, exist_ok=True)

for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)

def _headline(result, primary_col):
    top = result['ranking'].iloc[0]
    return {
        'champion': top['model'],
        'test_primary': float(top[primary_col]),
        'test_ci95': [float(top['test_ci95_low']), float(top['test_ci95_high'])],
    }

summary = {
    'sample_rows': MAX_ROWS,
    'cv_splits': config.cv_splits,
    'bootstrap_iterations': config.bootstrap_iterations,
    'selection_criterion': config.selection_criterion,
    'geo_coverage': float(coverage),
    'regression': _headline(reg, 'test_MAE'),
    'classification': _headline(cls, 'test_ROC_AUC'),
}
(output / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary

{'sample_rows': 15000,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'selection_criterion': 'worst',
 'geo_coverage': 0.9872889452036886,
 'regression': {'champion': 'LinearSVR (SVM)',
  'test_primary': 4.611699514771404,
  'test_ci95': [4.467956410917454, 4.744079044984438]},
 'classification': {'champion': 'Random Forest (ensamble)',
  'test_primary': 0.5119940140897168,
  'test_ci95': [0.4707802393631849, 0.5515316868556699]}}

## 7. Cierre

El experimento cumple su rol metodológico: documenta que la elección del criterio de selección (media vs. peor pliegue de CV) no modifica los campeones en este dataset. Con señales previas a la compra el techo queda:

- **Regresión:** LinearSVR, MAE 4.61 días en test.
- **Clasificación:** Random Forest por CV; Logística es el modelo que efectivamente generaliza a la ventana de test (ROC-AUC 0.734). La discrepancia entre ambos es el shift temporal, no el criterio de selección.

Para mover más el resultado habría que poner el holdout de validación más cerca del test en el tiempo o aplicar una corrección de shift explícita (reponderación por importancia, adaptación de dominio). Esto excede el alcance declarado del Entregable II y se deja como limitación en el informe.